# Colab Training Notebook

This notebook is a Colab-ready runner for this project.

It follows the same structure as local training:
1. Set project path
2. Install dependencies
3. Prepare data (`data/get_data.py` + `utils/data_edit.py`)
4. (Optional) compare parameter counts
5. Train `adaptive`, `matched`, and `base` models
6. Inspect checkpoints

## 1) Set project directory

If you already uploaded this repo to Colab runtime, set `PROJECT_DIR` accordingly.

If not, clone your GitHub repo first (see optional cell below).

In [ ]:
import os

PROJECT_DIR = "/content/a-gpt"  # <- change if your repo path is different
if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(
        f"Project dir not found: {PROJECT_DIR}. "
        "Upload/clone your repo first, then rerun."
    )

%cd {PROJECT_DIR}
print("Using project:", os.getcwd())

In [ ]:
# Optional: clone from GitHub if project is not present
# REPO_URL = "https://github.com/<your-username>/<your-repo>.git"
# !git clone "$REPO_URL" /content/a-gpt

## 2) Install dependencies and verify GPU

In [ ]:
!pip -q install torch tiktoken datasets tqdm wandb huggingface_hub

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 3) Optional: W&B login

In [ ]:
USE_WANDB = False  # set True to enable wandb logging
WANDB_PROJECT = "gpt-pretrain"
WANDB_API_KEY = ""  # <- paste key or keep empty to use interactive login

if USE_WANDB:
    import os
    import wandb

    if WANDB_API_KEY:
        os.environ["WANDB_API_KEY"] = WANDB_API_KEY
        wandb.login(key=WANDB_API_KEY)
    else:
        wandb.login()

## 4) Create and prepare data

This runs:
- `data/get_data.py` (downloads TinyStories + Cosmopedia wikihow text)
- `utils/data_edit.py` (truncates cosmopedia text to tiny text character length)

In [ ]:
!python data/get_data.py
!python utils/data_edit.py

!ls -lh data

## 5) (Optional) Compare model parameter counts

In [ ]:
!python -m utils.model_comparision

## 6) Train all three models (mixed pretraining)

This runs sequentially: `adaptive` -> `matched` -> `base`.

`MAX_STEPS` should align with your current experiment plan (e.g. 1000).

In [ ]:
import subprocess

MAX_STEPS = 1000
models = ["adaptive", "matched", "base"]

for model in models:
    cmd = [
        "python", "-m", "train.run",
        "--model", model,
        "--max_steps", str(MAX_STEPS),
    ]

    if USE_WANDB:
        cmd += [
            "--use_wandb",
            "--wandb_project", WANDB_PROJECT,
            "--wandb_run_name", f"{model}-mixed-v1",
        ]

    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

print("All training runs completed.")

## 7) Inspect checkpoints

In [ ]:
!ls -lh checkpoints

## 8) Optional: Hugging Face login

Use a token placeholder or interactive login, then upload checkpoints to HF Hub.

In [ ]:
HF_TOKEN = ""  # <- paste token (starts with hf_) or keep empty for interactive login

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
else:
    !huggingface-cli login

## 9) Optional: Push checkpoints to HF Hub

Set your target repo id, then run this cell to upload the full `checkpoints/` folder.

In [ ]:
from huggingface_hub import HfApi

HF_REPO_ID = "your-username/adaptive-gpt-mixed"  # <- change this

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True)
api.upload_folder(
    folder_path="checkpoints",
    repo_id=HF_REPO_ID,
    repo_type="model",
)

print(f"Uploaded checkpoints to: https://huggingface.co/{HF_REPO_ID}")